# Netflix User Behavior Segmentation

## Project Overview
This project analyzes Netflix user behavior by combining user profile data and watch history. The objective is to identify distinct viewer segments using unsupervised machine learning techniques.

## Objectives
- Explore user demographics and subscription patterns
- Engineer behavioral features from viewing history
- Cluster users using K-Means
- Interpret customer segments
- Apply Factor Analysis to uncover latent viewing behaviors

## Dataset
Located in `data/` (see `README.md` for the full data dictionary):
- `users.csv` — subscriber profiles and account details
- `watch_history.csv` — per-session viewing events

## Tools
Python • Pandas • NumPy • Scikit-learn • factor_analyzer • Matplotlib • Seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings

warnings.filterwarnings('ignore')


## 2. Load Data

In [ ]:
DATA_DIR = Path("data")

users = pd.read_csv(DATA_DIR / "users.csv")
watch_hist = pd.read_csv(DATA_DIR / "watch_history.csv")


## 3. Data Cleaning

In [ ]:
# Check Users

users.info()


In [ ]:
# Check missing values

print("\nMissing Values per Column:")
print(users.isnull().sum().sort_values(ascending=False))


In [ ]:
# Get value counts for variables of interest

# Columns of interest
cols = [
    'gender',
    'country',
    'state_province',
    'city',
    'subscription_plan',
    'is_active',
    'primary_device',
    'household_size',
]

# Loop through and print value counts
for col in cols:
    print(f"\n--- Value Counts for '{col}' ---")
    print(users[col].value_counts(dropna=False))


In [ ]:
# Histogram of age

users['age'].hist()
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Users")
plt.tight_layout()
plt.show()


In [ ]:
# Get counts per gender

if "gender" in users.columns:
    plt.figure(figsize=(6, 4))
    users['gender'].value_counts(dropna=False).plot(kind='bar')
    plt.title("Gender Distribution")
    plt.tight_layout()
    plt.show()


In [ ]:
# Get counts per Subscription Plan

if "subscription_plan" in users.columns:
    plt.figure(figsize=(6, 4))
    users['subscription_plan'].value_counts().plot(kind='bar')
    plt.title("Subscription Plan Counts")
    plt.tight_layout()
    plt.show()


### Distribution of Monthly Spend

Before clustering, we inspect the spending distribution to identify skewness and potential outliers.

In [ ]:
# Get distribution of monthly spend

if "monthly_spend" in users.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(users['monthly_spend'], kde=True)
    plt.title("Monthly Spend Distribution")
    plt.tight_layout()
    plt.savefig('monthly_spend.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Apply logarithmic transformation to treat outliers

users['monthly_spend_log'] = np.log1p(users['monthly_spend'])


In [ ]:
# Drop original monthly spend column

new_users = users.drop('monthly_spend', axis=1)


### Distribution of Device and Plans

In [ ]:
# Get counts per device

if "primary_device" in users.columns:
    plt.figure(figsize=(6, 4))
    users['primary_device'].value_counts().plot(kind='bar')
    plt.title("Primary Device Usage")
    plt.tight_layout()
    plt.show()


In [ ]:
# Active vs inactive by plan

if "is_active" in users.columns and "subscription_plan" in users.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=users, x="subscription_plan", hue="is_active")
    plt.title("Activity Status by Subscription Plan")
    plt.tight_layout()
    plt.show()


## 4. Exploratory Data Analysis

In [ ]:
# Age vs monthly spend

if "age" in users.columns and "monthly_spend_log" in users.columns:
    plt.figure(figsize=(8, 4))
    sns.scatterplot(data=users, x="age", y="monthly_spend_log")
    plt.title("Age vs Monthly Spend")
    plt.tight_layout()
    plt.show()


In [ ]:
# Signups over time

users['subscription_start_date'] = pd.to_datetime(
    users['subscription_start_date'],
    errors='coerce'
)

if "subscription_start_date" in users.columns:
    users['signup_month'] = users['subscription_start_date'].dt.to_period('M')
    plt.figure(figsize=(10, 4))
    users['signup_month'].value_counts().sort_index().plot()
    plt.title("User Signups Over Time")
    plt.xlabel("Month")
    plt.ylabel("Users")
    plt.tight_layout()
    plt.show()


In [ ]:
# Check watch history

watch_hist.info()


In [ ]:
# Merge users and watch history on user_id

merged_users_watch_hist = watch_hist.merge(new_users, on="user_id", how="left")


In [ ]:
merged_users_watch_hist.info()
merged_users_watch_hist.head()


## 5. Feature Engineering

### Behavioral Features

We will create the following user-level metrics:

- Average session duration
- Total watch time
- Unique titles watched
- Watch frequency
- Subscription tenure
- Search rate
- Binge score

In [ ]:
# Convert date columns
merged_users_watch_hist['watch_date'] = pd.to_datetime(merged_users_watch_hist['watch_date'])
merged_users_watch_hist['created_at'] = pd.to_datetime(merged_users_watch_hist['created_at'])

# Extract time-based features
merged_users_watch_hist['hour'] = merged_users_watch_hist['watch_date'].dt.hour
merged_users_watch_hist['day_of_week'] = merged_users_watch_hist['watch_date'].dt.dayofweek
merged_users_watch_hist['is_weekend'] = merged_users_watch_hist['day_of_week'].isin([5, 6]).astype(int)

# Aggregate features by user
user_features = merged_users_watch_hist.groupby('user_id').agg({
    # Viewing metrics
    'session_id': 'count',  # Total sessions
    'watch_duration_minutes': ['mean', 'sum', 'std'],
    'progress_percentage': ['mean', 'std'],
    'movie_id': 'nunique',  # Unique content watched

    # Engagement metrics
    'user_rating': ['count', 'mean'],  # Rating behavior
    'is_download': 'mean',  # Download rate

    # Device usage
    'device_type': lambda x: x.nunique(),  # Device diversity
    'primary_device': lambda x: x.mode()[0] if len(x.mode()) > 0 else None,

    # Quality preferences
    'quality': lambda x: (x == 'HD').mean(),  # HD preference

    # Time patterns
    'hour': ['mean', 'std'],
    'is_weekend': 'mean',

    # User demographics
    'age': 'first',
    'gender': 'first',
    'subscription_plan': 'first',
    'household_size': 'first',
    'subscription_start_date': 'first',
    'monthly_spend_log': 'first',
}).reset_index()

# Flatten column names
user_features.columns = ['_'.join(col).strip('_') if col[1] else col[0]
                         for col in user_features.columns.values]

print(f"\nUser-level features created: {user_features.shape}")
print("\nFeature columns:")
print(user_features.columns.tolist())


In [ ]:
# Engineer additional behavioral features
user_features['avg_session_duration'] = user_features['watch_duration_minutes_mean']
user_features['total_watch_time'] = user_features['watch_duration_minutes_sum']
user_features['watch_consistency'] = user_features['watch_duration_minutes_std']
user_features['completion_rate'] = user_features['progress_percentage_mean']
user_features['unique_content_ratio'] = user_features['movie_id_nunique'] / user_features['session_id_count']
user_features['rating_engagement'] = user_features['user_rating_count'] / user_features['session_id_count']
user_features['download_preference'] = user_features['is_download_mean']
user_features['hd_preference'] = user_features['quality_<lambda>']
user_features['device_diversity'] = user_features['device_type_<lambda>']
user_features['weekend_viewer'] = user_features['is_weekend_mean']
user_features['peak_hour'] = user_features['hour_mean']


In [ ]:
print(user_features.columns)


In [ ]:
plt.style.use('fivethirtyeight')  # A pleasant style for visualizations

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Distribution of Key Behavioral Features', fontsize=18)

# --- Plot 1: Unique Content Ratio ---
sns.histplot(user_features['unique_content_ratio'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('A) Unique Content Ratio (Content Diversity)', fontsize=14)
axes[0].set_xlabel('Unique Movies / Total Sessions')
axes[0].set_ylabel('Number of Users')
axes[0].axvline(user_features['unique_content_ratio'].median(), color='red', linestyle='--', label=f"Median: {user_features['unique_content_ratio'].median():.2f}")
axes[0].legend()

# --- Plot 2: Completion Rate ---
sns.histplot(user_features['completion_rate'], kde=True, ax=axes[1], color='lightcoral')
axes[1].set_title('B) Completion Rate (Attention/Interest)', fontsize=14)
axes[1].set_xlabel('Average Progress Percentage (0.0 to 1.0)')
axes[1].set_ylabel('Number of Users')
axes[1].axvline(user_features['completion_rate'].median(), color='red', linestyle='--', label=f"Median: {user_features['completion_rate'].median():.2f}")
axes[1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to prevent title overlap
plt.show()


In [ ]:
# Calculate subscription tenure (in days)

user_features['subscription_tenure'] = (pd.Timestamp.now() -
                                       pd.to_datetime(user_features['subscription_start_date_first'])).dt.days


In [ ]:
# Binge-watching score: high watch time + high sessions + low unique content ratio

user_features['binge_score'] = (
    user_features['avg_session_duration'] *
    user_features['session_id_count'] /
    (user_features['unique_content_ratio'] + 1)
)


In [ ]:
# Create action-based features

action_features = merged_users_watch_hist.groupby(['user_id', 'action']).size().unstack(fill_value=0)
action_features = action_features.div(action_features.sum(axis=1), axis=0)  # Normalize

# Merge action features
user_features = user_features.merge(action_features, on='user_id', how='left')


In [ ]:
# Search rate (if 'search' is an action)
if 'search' in action_features.columns:
    user_features['search_rate'] = action_features['search']
else:
    user_features['search_rate'] = 0

print(f"\nFinal feature set: {user_features.shape}")


## 6. Data Preparation

In [ ]:
print("\n" + "="*80)
print("DATA PREPARATION")
print("="*80)

# Select numerical features for clustering
numerical_features = [
    'session_id_count',
    'avg_session_duration',
    'total_watch_time',
    'watch_consistency',
    'completion_rate',
    'unique_content_ratio',
    'rating_engagement',
    'download_preference',
    'hd_preference',
    'device_diversity',
    'weekend_viewer',
    'peak_hour',
    'subscription_tenure',
    'binge_score',
    'search_rate',
    'age',
    'household_size',
    'monthly_spend_log'
]

# Filter features that exist in the dataframe
numerical_features = [f for f in numerical_features if f in user_features.columns]

# Create modeling dataset
X = user_features[numerical_features].copy()

# Handle missing values
print("\nMissing values before imputation:")
print(X.isnull().sum()[X.isnull().sum() > 0])

# Impute with median
for col in X.columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

print("\nFeatures for modeling:")
print(X.columns.tolist())
print(f"\nShape: {X.shape}")

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)


## 7. K-Means Clustering

K-Means clustering was selected because the objective is customer segmentation based on behavioral similarity.

Features were standardized to ensure equal weighting across variables.

In [ ]:
print("\n" + "="*80)
print("K-MEANS CLUSTERING")
print("="*80)

# Determine optimal number of clusters using elbow method and silhouette score
inertias = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot elbow and silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, 'o-', linewidth=2, markersize=8, color='steelblue')
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, silhouette_scores, 'o-', linewidth=2, markersize=8, color='darkgreen')
ax2.set_xlabel('Number of Clusters (k)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('clustering_optimization.png', dpi=300, bbox_inches='tight')
plt.show()

# Fit final model with optimal k
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add clusters to dataframe
user_features['cluster'] = clusters

print(f"\nOptimal number of clusters: {optimal_k}")
print(f"Silhouette Score: {silhouette_score(X_scaled, clusters):.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X_scaled, clusters):.4f}")

print("\nCluster Distribution:")
print(user_features['cluster'].value_counts().sort_index())


The elbow curve suggests that three clusters provide the best balance between model simplicity and within-cluster variance.

## 8. Cluster Interpretation

In [ ]:
print("\n" + "="*80)
print("CLUSTER PROFILING")
print("="*80)

# Profile each cluster
cluster_profiles = user_features.groupby('cluster')[numerical_features].mean()

print("\nCluster Profiles (Mean Values):")
print(cluster_profiles.round(2))

# Visualize cluster profiles
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

key_features = ['avg_session_duration', 'completion_rate', 'search_rate', 'binge_score']
existing_features = [f for f in key_features if f in cluster_profiles.columns]

for idx, feature in enumerate(existing_features[:4]):
    ax = axes[idx]
    cluster_profiles[feature].plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
    ax.set_title(f'{feature} by Cluster', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster', fontsize=10)
    ax.set_ylabel('Mean Value', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=300, bbox_inches='tight')
plt.show()

# Detailed cluster descriptions
for cluster_id in range(optimal_k):
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_id}")
    print(f"{'='*60}")

    cluster_data = user_features[user_features['cluster'] == cluster_id]
    print(f"Size: {len(cluster_data)} users ({len(cluster_data)/len(user_features)*100:.1f}%)")

    # Key characteristics
    print("\nKey Characteristics:")
    for feature in existing_features[:6]:
        mean_val = cluster_data[feature].mean()
        overall_mean = user_features[feature].mean()
        pct_diff = ((mean_val - overall_mean) / overall_mean) * 100
        print(f"  {feature}: {mean_val:.2f} ({pct_diff:+.1f}% vs overall)")


### Cluster 0 – Casual Viewers

- Low watch time
- Few sessions
- Mostly Basic subscription
- High churn risk

**Recommendation:** Offer promotional upgrades and personalized recommendations.

## 9. Factor Analysis

K-Means gives us discrete segments, but it treats all behavioral features as equally important and independent. Factor Analysis instead looks for a small number of **latent factors** that explain the correlations among the 13 behavioral features below, which can be used for richer interpretation or as inputs to downstream models.

The workflow is:
1. Build the behavioral feature matrix
2. Check data quality (missing/infinite values, zero-variance features) and drop anything unsuitable
3. Test whether the data is suitable for Factor Analysis (Bartlett's test, KMO)
4. Determine the number of factors to extract (scree plot, Kaiser criterion, parallel analysis)
5. Fit the final model with Varimax rotation and interpret the factors

In [ ]:
# Install the factor-analysis library (only needed once per environment)
%pip install -q factor-analyzer

from factor_analyzer import FactorAnalyzer, calculate_bartlett_sphericity, calculate_kmo


In [ ]:
# Candidate behavioral features for Factor Analysis
behavioral_features = [
    # Viewing Intensity
    'avg_session_duration',   # How long they watch
    'total_watch_time',       # Total consumption
    'session_id_count',       # Frequency of use

    # Content Engagement
    'completion_rate',        # Do they finish content?
    'binge_score',             # Consecutive viewing
    'unique_content_ratio',   # Content variety

    # Discovery Behavior
    'search_rate',            # Active searching
    'device_diversity',       # Platform exploration

    # Quality & Preferences
    'hd_preference',           # Quality consciousness
    'download_preference',    # Offline viewing
    'rating_engagement',      # Interactive behavior

    # Temporal Patterns
    'weekend_viewer',          # Time preferences
    'watch_consistency',      # Regular vs sporadic
]

X = user_features[behavioral_features].copy()
X = X.fillna(X.median())

print(f"\nBehavioral Features: {X.shape[1]} features, {X.shape[0]} users")
print("\nFeatures included:")
for i, feat in enumerate(X.columns, 1):
    print(f"  {i}. {feat}")


In [ ]:
# Data quality checks before fitting Factor Analysis

print("NaN count per feature:")
print(X.isnull().sum())

print("\nInfinite value count per feature:")
print(np.isinf(X).sum())

print("\nFeature variance:")
print(X.var())


`search_rate` has (close to) zero variance, which breaks the correlation matrix Factor Analysis relies on. We drop it before scaling and testing suitability.

In [ ]:
# Drop the zero-variance feature and re-scale
low_variance_features = ['search_rate']
X_cleaned = X.drop(columns=low_variance_features, errors='ignore')

print(f"Features remaining for Factor Analysis: {len(X_cleaned.columns)}")

scaler = StandardScaler()
X_scaled_cleaned = scaler.fit_transform(X_cleaned)
X_scaled_cleaned = pd.DataFrame(X_scaled_cleaned, columns=X_cleaned.columns)


In [ ]:
print("\n" + "="*80)
print("TESTING DATA SUITABILITY FOR FACTOR ANALYSIS")
print("="*80)

# Bartlett's Test of Sphericity: tests if the correlation matrix differs
# significantly from an identity matrix. p < 0.05 => suitable for FA.
chi_square_value, p_value_bartlett = calculate_bartlett_sphericity(X_scaled_cleaned)
print(f"\n1. Bartlett's Test of Sphericity: (p-value = {p_value_bartlett:.4f})")
if p_value_bartlett < 0.05:
    print("   ✓ Data is suitable (p < 0.05). Correlation exists.")
else:
    print("   ✗ Data is not suitable (p >= 0.05).")

# Kaiser-Meyer-Olkin (KMO) Test: measures sampling adequacy. KMO >= 0.6 is acceptable.
kmo_all, kmo_model = calculate_kmo(X_scaled_cleaned)
print(f"\n2. Kaiser-Meyer-Olkin (KMO) Test: (Overall KMO = {kmo_model:.4f})")
if kmo_model >= 0.6:
    print("   ✓ Good sampling adequacy (KMO >= 0.6).")
else:
    print("   ✗ Poor sampling adequacy.")


In [ ]:
print("\n" + "="*80)
print("DETERMINING OPTIMAL NUMBER OF FACTORS")
print("="*80)

# Fit an unrotated FactorAnalyzer to get all eigenvalues
n_features = X_cleaned.shape[1]
fa_temp = FactorAnalyzer(n_factors=n_features, rotation=None)
fa_temp.fit(X_scaled_cleaned)
eigenvalues, _ = fa_temp.get_eigenvalues()

# Kaiser criterion: keep factors with eigenvalue > 1
optimal_factors_kaiser = sum(eigenvalues > 1)
print(f"Kaiser Criterion: {optimal_factors_kaiser} factors have an eigenvalue > 1.")

# Parallel analysis: compare against eigenvalues from random (uncorrelated) data
n_iterations = 100
random_eigenvalues = np.zeros((n_iterations, n_features))
for i in range(n_iterations):
    random_data = np.random.normal(size=X_scaled_cleaned.shape)
    fa_random = FactorAnalyzer(n_factors=n_features, rotation=None)
    fa_random.fit(random_data)
    random_eigenvalues[i], _ = fa_random.get_eigenvalues()

percentile_95 = np.percentile(random_eigenvalues, 95, axis=0)
optimal_factors_parallel = sum(eigenvalues > percentile_95)
print(f"Parallel Analysis: {optimal_factors_parallel} factors have an eigenvalue above the 95th percentile of random data.")

# Plot the scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_features + 1), eigenvalues, marker='o', linestyle='--', color='darkblue', label='Observed data')
plt.plot(range(1, n_features + 1), percentile_95, marker='x', linestyle=':', color='gray', label='95th percentile (random data)')
plt.axhline(1, color='red', linestyle='-', label='Kaiser Criterion (Eigenvalue = 1)')
plt.title('Scree Plot to Determine Number of Factors')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.xticks(range(1, n_features + 1))
plt.savefig('scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()


Both the Kaiser criterion and parallel analysis point to a similar range. Combined with a manual check of interpretability, **5 factors** were selected for the final model.

In [ ]:
# Fit the final Factor Analyzer with Varimax rotation
n_factors_final = 5

fa_final = FactorAnalyzer(n_factors=n_factors_final, rotation='varimax')
fa_final.fit(X_scaled_cleaned)

# Get the factor loadings matrix
loadings = pd.DataFrame(
    fa_final.loadings_,
    index=X_scaled_cleaned.columns,
    columns=[f'Factor_{i+1}' for i in range(n_factors_final)]
)


def highlight_loadings(loadings):
    """Highlight loadings with an absolute value above 0.4 (a common rule-of-thumb cutoff)."""
    return loadings.apply(
        lambda x: np.where(np.abs(x) > 0.4, 'background-color: lightgreen', None),
        axis=0
    )


print("="*85)
print(f"FACTOR LOADINGS MATRIX (Varimax Rotation, {n_factors_final} Factors)")
print("="*85)
loadings.style.apply(highlight_loadings, axis=None)


In [ ]:
# Identify the dominant features for each factor and suggest an interpretation
suggested_names = {}

for i in range(n_factors_final):
    factor_name = f'Factor_{i+1}'
    dominant = loadings[factor_name].abs().nlargest(3)

    print(f"\n{factor_name}:")
    print("  Top 3 features:")
    for feat, loading_val in zip(dominant.index, loadings.loc[dominant.index, factor_name]):
        print(f"    • {feat} ({loading_val:+.3f})")

    top_feature = dominant.index[0]
    if 'duration' in top_feature or 'watch_time' in top_feature:
        suggestion = "Engagement Intensity / Viewing Commitment"
    elif 'search' in top_feature or 'discovery' in top_feature:
        suggestion = "Content Discovery / Exploration Behavior"
    elif 'completion' in top_feature or 'binge' in top_feature:
        suggestion = "Content Completion / Binge-Watching Tendency"
    elif 'quality' in top_feature or 'hd' in top_feature:
        suggestion = "Quality Consciousness / Premium Preferences"
    elif 'device' in top_feature or 'platform' in top_feature:
        suggestion = "Multi-Platform Usage / Device Flexibility"
    elif 'weekend' in top_feature or 'temporal' in top_feature:
        suggestion = "Temporal Viewing Patterns"
    else:
        suggestion = "Behavior Pattern (needs interpretation)"

    suggested_names[factor_name] = suggestion
    print(f"  ➜ Suggested interpretation: {suggestion}")


In [ ]:
# Compute factor scores for each user
factor_scores = fa_final.transform(X_scaled_cleaned)
factor_scores_df = pd.DataFrame(
    factor_scores,
    columns=[f'Factor_{i+1}' for i in range(n_factors_final)],
    index=X_scaled_cleaned.index
)

print("Factor Scores (first 10 users):")
print(factor_scores_df.head(10).round(3))

print("\nFactor Score Statistics:")
print(factor_scores_df.describe().round(3))


In [ ]:
# Assess factor quality: communalities (variance explained per feature)
communalities = fa_final.get_communalities()
communality_df = pd.DataFrame({
    'Feature': X_scaled_cleaned.columns,
    'Communality': communalities
}).sort_values('Communality', ascending=False)

print("Communalities (variance explained per feature):")
print(communality_df.to_string(index=False))
print(f"\nAverage communality: {communalities.mean():.3f}")
print("✓ Good if > 0.5 (factor model explains most of the feature's variance)")

# With Varimax (orthogonal) rotation, factor correlations should be close to 0
print("\nFactor Correlations:")
print(factor_scores_df.corr().round(3))


In [ ]:
# Save results
loadings.to_csv('latent_factor_loadings.csv')
factor_scores_df.to_csv('user_factor_scores.csv')

interpretation_df = pd.DataFrame([
    {
        'Factor': k,
        'Interpretation': v,
        'Top_Features': ', '.join(loadings[k].abs().nlargest(3).index)
    }
    for k, v in suggested_names.items()
])
interpretation_df.to_csv('factor_interpretations.csv', index=False)

communality_df.to_csv('feature_communalities.csv', index=False)

print("Saved: latent_factor_loadings.csv, user_factor_scores.csv, factor_interpretations.csv, feature_communalities.csv")


### How to Use the Latent Factors

These factors represent underlying dimensions that influence user behavior, and can be used beyond this notebook:

1. **Clustering** — use factor scores instead of raw features to reduce noise and improve cluster quality (e.g. `kmeans.fit(factor_scores)`).
2. **Segmentation** — combine factor scores directly, e.g. *high Factor 1 + low Factor 2 = "Engaged but not explorers"*.
3. **Prediction** — use factor scores as compact, decorrelated inputs to churn or recommendation models.
4. **Reporting** — communicate a handful of named behavior types instead of 13+ individual features.
5. **Personalization** — tailor content recommendations to a user's factor profile (e.g. a high "Discovery" factor → surface more diverse genres).

**Next steps:** review `factor_interpretations.csv` and assign final names to each factor, examine users with extreme factor scores, validate the factors with domain experts, and track how factor scores evolve over time.